# Graph Model Features with Scikit-Learn
There may be times that typical graph models may not produce sufficient models. To combat this, graph features can be utilized as the feature vector for traditional machine learning (ML) models from Scikit-Learn. To do this, the molecules will be converted into molecular graphs. Then the graph latent space will be exported. These features are in numpy format, which can then be used to build traditional ML models.

This tutorial will demo how to create four different molecular graph features and output the latent space. The first example will also show how these features can be used to build a RandomForrest classifier model and predict sample molecules.

## Import packages
The four graph models include Graph Convolutional Neural Network (GCNN), Graph Isomorphism Network (GINE), Graph Sage (GSAGE), and Graph Attention Networks (GAT). These classes are imported from teh **latent_space** module.

In [1]:
import pandas as pd
from tqdm import tqdm
from mminer_graphs import MolToGraph, EarlyStopping
from mminer_graphs.model.latent_space import GCNN, GINE, GSAGE, GAT

## Dataset Prep
Similar to the other tutorials, the bace dataset will be used.

In [2]:
data = pd.read_csv("dataset/bace.csv")
cols = ['CID', 'smiles', 'Class', 'Model']
dataset = data[cols]
print(f"The shape of the dataset: {dataset.shape}")
dataset.head()

The shape of the dataset: (1513, 4)


,CID,smiles,Class,Model
0,BACE_1,O1CC[C@@H](NC(=O)[C@@H](Cc2cc3cc(ccc3nc2N)-c2c...,1,train
1,BACE_2,Fc1cc(cc(F)c1)C[C@H](NC(=O)[C@@H](N1CC[C@](NC(...,1,train
2,BACE_3,S1(=O)(=O)N(c2cc(cc3c2n(cc3CC)CC1)C(=O)N[C@H](...,1,train
3,BACE_4,S1(=O)(=O)C[C@@H](Cc2cc(O[C@H](COCC)C(F)(F)F)c...,1,train
4,BACE_5,S1(=O)(=O)N(c2cc(cc3c2n(cc3CC)CC1)C(=O)N[C@H](...,1,train


The dataset is separated by train/test labels. The molecule smiles string and class label will be extracted from the table and, in a list, wil lbe converted into molecular graph data. The first five entries in train_graphs will be printed to quickly visualize the data.

In [3]:
train = dataset[dataset.iloc[:, -1] == 'train']
test = dataset[dataset.iloc[:, -1] == 'test']

train_graphs = [MolToGraph(smi=smi, label=y).graph_data for smi, y in tqdm(zip(train.smiles, train.Class), desc="Converting Train Graphs")]
test_graphs = [MolToGraph(smi=smi, label=y).graph_data for smi, y in tqdm(zip(test.smiles, test.Class), desc="Converting Test Graphs")]

train_graphs[:5]

Converting Train Graphs: 1376it [00:02, 607.80it/s]
Converting Test Graphs: 137it [00:00, 602.64it/s]


[Data(x=[32, 142], edge_index=[2, 70], edge_attr=[70, 16], y=[1]),
 Data(x=[47, 142], edge_index=[2, 100], edge_attr=[100, 16], y=[1]),
 Data(x=[42, 142], edge_index=[2, 92], edge_attr=[92, 16], y=[1]),
 Data(x=[40, 142], edge_index=[2, 84], edge_attr=[84, 16], y=[1]),
 Data(x=[44, 142], edge_index=[2, 96], edge_attr=[96, 16], y=[1])]

## GCNN
First, the GCNN model wil lbe used. By dfault, it will utilize AtomGraph features. Printing the model will show the graph model structure. In this example, three graph layers are used with an output of 128 dimensions.

In [4]:
graph_model = GCNN(node_dim=142, hidden_dim=128, graph_layers=3, reduce_graph=False)
graph_model

GCNN(
  (gcn_conv): ModuleList(
    (0): GCNConv(142, 128)
    (1-2): 2 x GCNConv(128, 128)
  )
)

### Fitting the Graph Model
A graph model will need to trained to ensure that the weights in message-passing layers are updated and captures patterns from the molecular graph. In this example, EarlyStopping is also shown and is used similarly as in the other tutorials. The graph model will be trained using the **.fit** method.

In [5]:
estop = EarlyStopping(monitor='val_loss', patience=350, mode='min')

graph_model.fit(train_graphs, test_graphs, epochs=350, optimizer="adam", device='cpu', batch_size=1024, estop=estop)


Model and Tensor running on cpu


Epoch: 100%|██████████| 350/350 [01:11<00:00,  4.92it/s]


### Extract Train/Test Data

Once the graph model has been trained, the graph latent space can be extracted. This is done using the **train_graph_space** and will return train_x, train_y, test_x, and test_y data. This is helpful as it can be used to feed data directly into sklearn models.

In [6]:
train_x, train_y, test_x, test_y = graph_model.train_graph_space(train_graphs, test_graphs, batch_size=1024)

Converting test data: 100%|██████████| 1/1 [00:00<00:00, 73.19it/s]


Visaulize the latent space vector. In this case, the vector for the training data is used.

In [7]:
train_x

array([[12.278871, 11.344118,  0.      , ...,  0.      ,  0.      ,
         0.      ],
       [12.598888, 11.355978,  0.      , ...,  0.      ,  0.      ,
         0.      ],
       [13.84058 , 11.647809,  0.      , ...,  0.      ,  0.      ,
         0.      ],
       ...,
       [13.665445, 11.462237,  0.      , ...,  0.      ,  0.      ,
         0.      ],
       [12.795818, 10.932955,  0.      , ...,  0.      ,  0.      ,
         0.      ],
       [12.153459, 13.166953,  0.      , ...,  0.      ,  0.      ,
         0.      ]], dtype=float32)

## Random Forest Model

This example shows builds a Random Forest Classifier model from the extracted graph latent space.

In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(train_x, train_y)

y_pred = rf.predict(test_x)
print("Accuracy:", accuracy_score(test_y, y_pred))

Accuracy: 0.7664233576642335


## Making Predictions

Making predictions requires a similar workflow. In this example, molecule smiles for predictions are placed in a list. Each smiles string is then converted into a molecular graph and placed inside another list. Finally, the latent space is generated using the **predict_graph_space** method.

**NOTE:** The molecules used for this example were taken from the training dataset above. Thus, the model has "cheated" and seen this data before. Future testing should be done using unseen data. 

In [9]:
# prediction
to_predict = ["O1CC[C@@H](NC(=O)[C@@H](Cc2cc3cc(ccc3nc2N)-c2ccccc2C)C)CC1(C)C", "Fc1cc(cc(F)c1)C[C@H](NC(=O)[C@@H](N1CC[C@](NC(=O)C)(CC(C)C)C1=O)CCc1ccccc1)[C@H](O)[C@@H]1[NH2+]C[C@H](OCCC)C1", "FC(F)(F)Oc1ccc(cc1)C1(N=C(N)N(C)C1=O)c1cc(ccc1)-c1ccc(OC)nc1"]
data = [MolToGraph(smi=smi).graph_data for smi in tqdm(to_predict, desc="Converting Data to Graphs")]

graph_latent_space = graph_model.predict_graph_space(data)

Converting data: 100%|██████████| 1/1 [00:00<00:00, 309.70it/s]


With the molecular graph features generated, the features can be given to the **.predict** method in the Random Forest Classifier. This will output an array containing the binary classificaiton for the molecules.

In [10]:
predicitons = rf.predict(graph_latent_space)
predicitons

array([1, 1, 0])